# Sidra Preferida — Análisis Completo XGBoost
## Flujo de trabajo (5 pasos)
1. **Comparar Caso 1, 2, 3 en mensual y semanal** → elegir el mejor
2. **Probar features de calendario** → ver cuál mejora el SMAPE
3. **Alimentar el notebook unificado XGBoost_5productos.ipynb**
4. **Métricas comparables con Tamara y Álvaro** (WAPE, MAPE, MAE%)
5. **Split fijo vs Walk-Forward** — comparar ambos métodos
> ⚠️ **Sidra especial:** el 90% de la demanda está en oct-nov-dic. Cualquier feature que toque esos meses puede interferir. Si ninguna feature mejora, la base es la mejor opción.


## 0. Imports y funciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import mean_absolute_error, mean_squared_error
try:
    import xgboost as xgb
    USE_XGB = True
    print("✅ XGBoost listo")
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor
    USE_XGB = False
    print("⚠️  Usando GradientBoostingRegressor")

SS = {2021: 4, 2022: 4, 2023: 4, 2024: 3, 2025: 4, 2026: 4}
print("✅ Imports OK")


In [ ]:
def construir_modelo():
    if USE_XGB:
        return xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                 subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                                 reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbosity=0)
    return GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                      max_depth=4, subsample=0.8, random_state=42)

def features_mensual(ts, extras=[]):
    df = pd.DataFrame({"y": ts.astype(float)}).reset_index()
    df.columns = ["fecha", "y"]
    df["mes_num"] = df["fecha"].dt.month
    df["trimestre"] = df["fecha"].dt.quarter
    df["año"] = df["fecha"].dt.year
    df["t"] = range(len(df))
    df["t2"] = df["t"] ** 2
    for lag in [1,2,3,4,6,12,13,24]:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    for v in [3,6,12]:
        df[f"roll_mean_{v}"] = df["y"].shift(1).rolling(v).mean()
        df[f"roll_std_{v}"]  = df["y"].shift(1).rolling(v).std()
        df[f"roll_max_{v}"]  = df["y"].shift(1).rolling(v).max()
    df["yoy_change"]  = df["y"].shift(1) - df["y"].shift(13)
    df["yoy_ratio"]   = df["y"].shift(1) / (df["y"].shift(13) + 1)
    df["ratio_trend"] = df["y"].shift(1).rolling(3).mean() / (df["y"].shift(1).rolling(12).mean() + 1)
    df["diff_1"] = df["y"].diff(1)
    for col_name, col_vals in extras:
        df[col_name] = col_vals
    return df

def features_semanal(ts, extras=[]):
    df = pd.DataFrame({"y": ts.astype(float)}).reset_index()
    df.columns = ["fecha", "y"]
    df["mes_num"]    = df["fecha"].dt.month
    df["semana_año"] = df["fecha"].dt.isocalendar().week.astype(int)
    df["trimestre"]  = df["fecha"].dt.quarter
    df["año"]        = df["fecha"].dt.year
    df["t"]          = range(len(df))
    df["t2"]         = df["t"] ** 2
    for lag in [1,2,3,4,8,13,26,52,53]:
        df[f"lag_{lag}"] = df["y"].shift(lag)
    for v in [4,8,13,26]:
        df[f"roll_mean_{v}"] = df["y"].shift(1).rolling(v).mean()
        df[f"roll_std_{v}"]  = df["y"].shift(1).rolling(v).std()
        df[f"roll_max_{v}"]  = df["y"].shift(1).rolling(v).max()
    df["yoy_change"]  = df["y"].shift(1) - df["y"].shift(53)
    df["yoy_ratio"]   = df["y"].shift(1) / (df["y"].shift(53) + 1)
    df["ratio_trend"] = df["y"].shift(1).rolling(4).mean() / (df["y"].shift(1).rolling(26).mean() + 1)
    df["diff_1"] = df["y"].diff(1)
    for col_name, col_vals in extras:
        df[col_name] = col_vals
    return df

def walk_forward(df_feat, min_train, ventana=36):
    fc = [c for c in df_feat.columns if c not in ["fecha","y"]]
    X = df_feat[fc].values; y = df_feat["y"].values; fechas = df_feat["fecha"].values
    actuals, preds, dates = [], [], []
    for i in range(min_train, len(df_feat)):
        start = max(0, i - ventana)
        model = construir_modelo()
        model.fit(X[start:i], y[start:i])
        preds.append(max(0.0, float(model.predict(X[i:i+1])[0])))
        actuals.append(float(y[i]))
        dates.append(pd.Timestamp(fechas[i]))
    return np.array(actuals), np.array(preds), dates

def split_fijo(df_feat, test_desde="2025-01-01"):
    """Train: todo antes de test_desde. Test: desde test_desde en adelante."""
    fc = [c for c in df_feat.columns if c not in ["fecha","y"]]
    mask_train = df_feat["fecha"] < test_desde
    mask_test  = df_feat["fecha"] >= test_desde
    X_train = df_feat[fc][mask_train].values
    y_train = df_feat["y"][mask_train].values
    X_test  = df_feat[fc][mask_test].values
    y_test  = df_feat["y"][mask_test].values
    fechas_test = df_feat["fecha"][mask_test].values
    if len(X_train) == 0 or len(X_test) == 0:
        return np.array([]), np.array([]), []
    model = construir_modelo()
    model.fit(X_train, y_train)
    preds = np.maximum(0, model.predict(X_test))
    return y_test, preds, [pd.Timestamp(f) for f in fechas_test]

def calcular_metricas(actuals, preds, label=""):
    if len(actuals) == 0:
        return dict(label=label, MAE=0, RMSE=0, MAPE=0, SMAPE=0,
                    WAPE=0, MAE_pct=0, Bias=0, R2=float("nan"), n=0)
    mae   = mean_absolute_error(actuals, preds)
    rmse  = np.sqrt(mean_squared_error(actuals, preds))
    mask  = actuals > 0
    mape  = np.mean(np.abs((actuals[mask]-preds[mask])/actuals[mask]))*100 if mask.any() else 0
    smape = np.mean(2*np.abs(actuals-preds)/(np.abs(actuals)+np.abs(preds)+1e-9))*100
    wape  = np.sum(np.abs(actuals-preds)) / np.sum(actuals) * 100 if np.sum(actuals) > 0 else 0
    mae_pct = mae / np.mean(actuals) * 100 if np.mean(actuals) > 0 else 0
    bias  = np.mean(preds - actuals)
    ss_r  = np.sum((actuals-preds)**2)
    ss_t  = np.sum((actuals-actuals.mean())**2)
    r2    = 1 - ss_r/ss_t if ss_t > 0 else float("nan")
    return dict(label=label, MAE=mae, RMSE=rmse, MAPE=mape, SMAPE=smape,
                WAPE=wape, MAE_pct=mae_pct, Bias=bias, R2=r2, n=len(actuals))

def imprimir_metricas(m):
    print(f"  SMAPE  : {m['SMAPE']:>7.2f}%  ← métrica principal")
    print(f"  WAPE   : {m['WAPE']:>7.2f}%  ← comparable con Tamara/Álvaro")
    print(f"  MAPE   : {m['MAPE']:>7.2f}%")
    print(f"  MAE    : {m['MAE']:>8.1f}  unidades/período")
    print(f"  MAE%   : {m['MAE_pct']:>7.2f}%  ← MAE como % de demanda media")
    print(f"  Bias   : {m['Bias']:>+8.1f}  (+ sobreestima / - subestima)")
    print(f"  R²     : {m['R2']:>8.3f}")
    print(f"  N      : {m['n']:>8}  períodos evaluados")

print("✅ Funciones listas")


## PASO 1 — Comparar Caso 1, 2, 3 en mensual y semanal
Esto replica lo que hiciste con Café Belén y Choclo Norte.
El caso con menor SMAPE mensual es el que usamos en el Paso 2.


In [ ]:
# ─── Cargar datos mensuales ───────────────────────────────
df_raw = pd.read_excel("SidraTotal.xlsx", sheet_name="Datos Producto")
df_raw["Mes"] = pd.to_datetime(df_raw["Mes"])
df_raw = df_raw[df_raw["Producto"] == 181913006].copy()
df_raw = df_raw.sort_values(["NomClienteAlter","Mes"]).reset_index(drop=True)

print(f"Mensual — registros: {len(df_raw)} | neg: {(df_raw['Demanda']<0).sum()} | ceros: {(df_raw['Demanda']==0).sum()}")
print(f"Rango: {df_raw['Mes'].min().date()} → {df_raw['Mes'].max().date()}")

# ─── Cargar datos semanales ──────────────────────────────
df_sem = pd.read_csv("Producto_181913006_semanal.csv")
df_sem["Fecha"] = pd.to_datetime(df_sem["Fecha"])
df_sem = df_sem.sort_values("Fecha").reset_index(drop=True)

print(f"Semanal  — semanas: {len(df_sem)} | neg: {(df_sem['Demanda']<0).sum()} | ceros: {(df_sem['Demanda']==0).sum()}")
print(f"Rango semanal: {df_sem['Fecha'].min().date()} → {df_sem['Fecha'].max().date()}")

# ─── Outliers IQR ────────────────────────────────────────
ts_check = df_raw.groupby("Mes")["Demanda"].sum().clip(lower=0)
Q1=ts_check.quantile(0.25); Q3=ts_check.quantile(0.75); IQR=Q3-Q1
LIM_IQR = float(Q3 + 1.5 * IQR)
out = ts_check[ts_check > LIM_IQR]
print(f"\nLímite IQR: {LIM_IQR:.0f} | Outliers: {len(out)}")
if len(out):
    print(f"  Meses: {list(out.index.strftime('%Y-%m'))}")
    print(f"  → 11 meses sobre el límite IQR — TODOS son oct-nov-dic. Son estacionalidad real, NO outliers. NO se capean.")
else:
    print("  → Sin outliers — NO se aplica capeo")


In [ ]:
def preprocesar_mensual(df, caso):
    d = df.copy().sort_values(["NomClienteAlter","Mes"])
    if caso == 3:
        for idx in d[d["Demanda"]<0].index:
            cli=d.loc[idx,"NomClienteAlter"]; mes_d=d.loc[idx,"Mes"]; dev=d.loc[idx,"Demanda"]
            prev=d[(d["NomClienteAlter"]==cli)&(d["Mes"]<mes_d)]
            if not prev.empty:
                pi=prev.index[-1]; d.loc[pi,"Demanda"]=max(0,d.loc[pi,"Demanda"]+dev)
            d.loc[idx,"Demanda"]=0
    else:
        d["Demanda"]=d["Demanda"].clip(lower=0)
        if caso==2: d=d[d["Demanda"]>0]
    ts=d.groupby("Mes")["Demanda"].sum().astype(float)
    ts=ts.reindex(pd.date_range(ts.index.min(),ts.index.max(),freq="MS")).fillna(0.0)
    return ts

def preprocesar_semanal(df, caso):
    d = df.copy()
    if caso==1:
        ts=d.set_index("Fecha")["Demanda"].clip(lower=0)
    elif caso==2:
        ts=d.set_index("Fecha")["Demanda"].clip(lower=0)
        ts=ts[ts>0].reindex(pd.date_range(d["Fecha"].min(),d["Fecha"].max(),freq="W-MON")).fillna(0)
        ts=ts[ts>0]
    else:
        ts=d.set_index("Fecha")["Demanda"].clip(lower=0)
        vals=ts.values.copy()
        for i in range(1,len(vals)-1):
            if vals[i]==0 and vals[i-1]>0 and vals[i+1]>0:
                vals[i]=(vals[i-1]+vals[i+1])/2
        ts=pd.Series(vals,index=ts.index)
    return ts

print("✅ Funciones de preprocessing listas")


In [ ]:
# ─── Correr los 6 casos ───────────────────────────────────
casos_mensual = {}
for caso in [1, 2, 3]:
    ts = preprocesar_mensual(df_raw, caso)
    df_f = features_mensual(ts).dropna().reset_index(drop=True)
    a, p, d = walk_forward(df_f, min_train=24, ventana=36)
    m = calcular_metricas(a, p, f"Caso {caso} Mensual")
    casos_mensual[f"Caso {caso} Mensual"] = {"m": m, "a": a, "p": p, "d": d}
    print(f"Caso {caso} Mensual → SMAPE: {m['SMAPE']:.2f}% | Bias: {m['Bias']:+.1f}")

print()
casos_semanal = {}
for caso in [1, 2, 3]:
    ts = preprocesar_semanal(df_sem, caso)
    df_f = features_semanal(ts).dropna().reset_index(drop=True)
    a, p, d = walk_forward(df_f, min_train=65, ventana=104)
    m = calcular_metricas(a, p, f"Caso {caso} Semanal")
    casos_semanal[f"Caso {caso} Semanal"] = {"m": m, "a": a, "p": p, "d": d}
    print(f"Caso {caso} Semanal → SMAPE: {m['SMAPE']:.2f}% | Bias: {m['Bias']:+.1f}")


In [ ]:
# ─── Tabla comparativa ────────────────────────────────────
todos = [v["m"] for v in {**casos_mensual, **casos_semanal}.values()]
df_res = pd.DataFrame(todos)[["label","SMAPE","WAPE","MAE","MAE_pct","Bias","R2","n"]]
df_res = df_res.sort_values("SMAPE").reset_index(drop=True)
df_res.index += 1

print("=" * 75)
print("  RANKING — menor SMAPE = mejor")
print("=" * 75)
print(df_res.to_string(float_format=lambda x: f"{x:.2f}"))
print("=" * 75)

ganador = df_res.iloc[0]
print(f"\n🏆 GANADOR: {ganador['label']}")
print(f"   SMAPE: {ganador['SMAPE']:.2f}% | WAPE: {ganador['WAPE']:.2f}% | Bias: {ganador['Bias']:+.1f}")

smape_m = df_res[df_res["label"].str.contains("Mensual")]["SMAPE"].mean()
smape_s = df_res[df_res["label"].str.contains("Semanal")]["SMAPE"].mean()
print(f"\n  Mensual promedio: {smape_m:.2f}%")
print(f"  Semanal promedio: {smape_s:.2f}%")
if smape_m < smape_s:
    print(f"  ✅ Mensual gana por {smape_s - smape_m:.2f} puntos de SMAPE")
else:
    print(f"  ✅ Semanal gana por {smape_m - smape_s:.2f} puntos de SMAPE")


In [ ]:
# ─── Gráfico heatmap ─────────────────────────────────────
try:
    import seaborn as sns
    pivot = df_res.set_index("label")["SMAPE"].reset_index()
    pivot["Frecuencia"] = pivot["label"].apply(lambda x: "Mensual" if "Mensual" in x else "Semanal")
    pivot["Caso"] = pivot["label"].apply(lambda x: x.split()[0]+" "+x.split()[1])
    piv = pivot.pivot(index="Caso", columns="Frecuencia", values="SMAPE")
    fig, ax = plt.subplots(figsize=(6,4))
    sns.heatmap(piv, annot=True, fmt=".1f", cmap="RdYlGn_r", linewidths=0.5,
                ax=ax, cbar_kws={"label":"SMAPE (%)"}, annot_kws={"size":12,"weight":"bold"})
    ax.set_title("SMAPE (%) — menor = mejor | Sidra Preferida", fontsize=11, fontweight="bold")
    plt.tight_layout(); plt.show()
except ImportError:
    print("Instalá seaborn para el heatmap: pip install seaborn")
    print(pivot.to_string())


## PASO 2 — Test de features con el caso ganador
Usamos el caso mensual ganador del Paso 1.
Cambiá `CASO_GANADOR` si el resultado fue distinto al esperado.
> ⚠️ **Sidra especial:** el 90% de la demanda está en oct-nov-dic. Cualquier feature que toque esos meses puede interferir. Si ninguna feature mejora, la base es la mejor opción.


In [ ]:
# ── Editá según el resultado del Paso 1 ──────────────────
CASO_GANADOR = 3  # ← cambiá por 1, 2 o 3 según el ganador

ts_ganador = preprocesar_mensual(df_raw, CASO_GANADOR)
print(f"Serie con Caso {CASO_GANADOR}: {len(ts_ganador)} meses | Promedio: {ts_ganador.mean():.0f}")

df0 = features_mensual(ts_ganador).dropna().reset_index(drop=True)
a0, p0, d0 = walk_forward(df0, min_train=24)
m0 = calcular_metricas(a0, p0, "0. Base")
print("\nBase (sin features extra):")
imprimir_metricas(m0)
SMAPE_BASE = m0["SMAPE"]


### 4.1 — Temporada Sidra (oct-nov-dic = 1)
**Por qué:** El 90% de la demanda cae en estos 3 meses. Sin este flag el modelo intenta aprenderlo solo desde los lags.

In [ ]:
df_f = features_mensual(ts_ganador).copy()
df_f["temporada_sidra"] = df_f["fecha"].dt.month.isin([10, 11, 12]).astype(int)
df_f = df_f.dropna().reset_index(drop=True)
a1, p1, d1 = walk_forward(df_f, min_train=24)
m1 = calcular_metricas(a1, p1, "1. + Temporada Sidra (oct-nov-dic = 1)")
print("=" * 50)
print("  1. + TEMPORADA SIDRA (OCT-NOV-DIC = 1)")
print("=" * 50)
imprimir_metricas(m1)
delta = SMAPE_BASE - m1["SMAPE"]
if delta > 0:   print(f"\n  ✅ MEJORA {delta:.2f}% → AGREGAR")
elif delta < 0: print(f"\n  ❌ EMPEORA {abs(delta):.2f}% → DESCARTAR")
else:           print(f"\n  🔵 Sin cambio")


### 4.2 — Navidad (nov-dic = 1)
**Por qué:** Subset de temporada sidra — probamos si un flag más acotado funciona mejor.

In [ ]:
df_f = features_mensual(ts_ganador).copy()
df_f["navidad"] = df_f["fecha"].dt.month.isin([11, 12]).astype(int)
df_f = df_f.dropna().reset_index(drop=True)
a2, p2, d2 = walk_forward(df_f, min_train=24)
m2 = calcular_metricas(a2, p2, "2. + Navidad (nov-dic = 1)")
print("=" * 50)
print("  2. + NAVIDAD (NOV-DIC = 1)")
print("=" * 50)
imprimir_metricas(m2)
delta = SMAPE_BASE - m2["SMAPE"]
if delta > 0:   print(f"\n  ✅ MEJORA {delta:.2f}% → AGREGAR")
elif delta < 0: print(f"\n  ❌ EMPEORA {abs(delta):.2f}% → DESCARTAR")
else:           print(f"\n  🔵 Sin cambio")


### 4.3 — Semana Santa (mes variable)
**Por qué:** Semana Santa puede tener pico — la gente consume más en reuniones familiares.

In [ ]:
df_f = features_mensual(ts_ganador).copy()
df_f["semana_santa"] = df_f.apply(lambda r: 1 if SS.get(r["fecha"].year)==r["fecha"].month else 0, axis=1)
df_f = df_f.dropna().reset_index(drop=True)
a3, p3, d3 = walk_forward(df_f, min_train=24)
m3 = calcular_metricas(a3, p3, "3. + Semana Santa (mes variable)")
print("=" * 50)
print("  3. + SEMANA SANTA (MES VARIABLE)")
print("=" * 50)
imprimir_metricas(m3)
delta = SMAPE_BASE - m3["SMAPE"]
if delta > 0:   print(f"\n  ✅ MEJORA {delta:.2f}% → AGREGAR")
elif delta < 0: print(f"\n  ❌ EMPEORA {abs(delta):.2f}% → DESCARTAR")
else:           print(f"\n  🔵 Sin cambio")


### 4.4 — Invierno paraguayo (jun-jul = 1)
**Por qué:** La sidra es una bebida fría — probamos si tiene algún patrón de invierno.

In [ ]:
df_f = features_mensual(ts_ganador).copy()
df_f["invierno"] = df_f["fecha"].dt.month.isin([6, 7]).astype(int)
df_f = df_f.dropna().reset_index(drop=True)
a4, p4, d4 = walk_forward(df_f, min_train=24)
m4 = calcular_metricas(a4, p4, "4. + Invierno paraguayo (jun-jul = 1)")
print("=" * 50)
print("  4. + INVIERNO PARAGUAYO (JUN-JUL = 1)")
print("=" * 50)
imprimir_metricas(m4)
delta = SMAPE_BASE - m4["SMAPE"]
if delta > 0:   print(f"\n  ✅ MEJORA {delta:.2f}% → AGREGAR")
elif delta < 0: print(f"\n  ❌ EMPEORA {abs(delta):.2f}% → DESCARTAR")
else:           print(f"\n  🔵 Sin cambio")


### Mejor combinación

In [ ]:
df5 = features_mensual(ts_ganador).copy()

# ── Descomentá solo las que MEJORARON ──
# df5["temporada_sidra"] = df5["fecha"].dt.month.isin([10, 11, 12]).astype(int)
# Descomentá la que haya mejorado
# Agregá o comentá según tus resultados del paso anterior

df5 = df5.dropna().reset_index(drop=True)
a5, p5, d5 = walk_forward(df5, min_train=24)
m5 = calcular_metricas(a5, p5, "5. Mejor combinación")
print("=" * 50); print("  MEJOR COMBINACIÓN"); print("=" * 50)
imprimir_metricas(m5)
delta = SMAPE_BASE - m5["SMAPE"]
if delta > 0:   print(f"\n  ✅ MEJORA total de {delta:.2f}% vs base")
elif delta < 0: print(f"\n  ❌ Empeora {abs(delta):.2f}% — revisá qué feature suma ruido")
else:           print(f"\n  🔵 Sin cambio")


In [ ]:
todos_f = [m0, m1, m2, m3, m4, m5]
df_feat_res = pd.DataFrame(todos_f)[["label","SMAPE","WAPE","MAE","Bias","R2"]]
df_feat_res = df_feat_res.sort_values("SMAPE").reset_index(drop=True)
df_feat_res.index += 1
print("=" * 65)
print("  RANKING FEATURES — menor SMAPE = mejor")
print("=" * 65)
print(df_feat_res.to_string(float_format=lambda x: f"{x:.2f}"))
print("=" * 65)
ganador_f = df_feat_res.iloc[0]
print(f"\n🏆 MEJOR FEATURE: {ganador_f['label']}")
print(f"   SMAPE: {ganador_f['SMAPE']:.2f}% | Mejora vs base: {SMAPE_BASE - ganador_f['SMAPE']:+.2f}%")


## PASO 3 — Resumen para XGBoost_5productos.ipynb

In [ ]:
m_final = m5  # ← cambiá por la mejor métrica (m0, m1... m5)

print("=" * 60)
print("  RESUMEN SIDRA PREFERIDA")
print("  → Copiá estos valores a XGBoost_5productos.ipynb")
print("=" * 60)
print(f"  Caso       : {CASO_GANADOR}")
print(f"  Feature    : {m_final['label']}")
print(f"  SMAPE      : {m_final['SMAPE']:.2f}%")
print(f"  WAPE       : {m_final['WAPE']:.2f}%")
print(f"  MAE        : {m_final['MAE']:.1f} unidades/mes")
print(f"  MAE%       : {m_final['MAE_pct']:.2f}%")
print(f"  Bias       : {m_final['Bias']:+.1f}")
print(f"  R²         : {m_final['R2']:.3f}")
print(f"  N          : {m_final['n']} meses evaluados")
if m_final["Bias"] < -50:
    print(f"\n  ⚠️  Factor de seguridad sugerido: +{abs(m_final['Bias']):.0f} u/mes")
print("=" * 60)


## PASO 4 — Métricas comparables con Tamara y Álvaro
Tamara y Álvaro usan **WAPE, MAPE y MAE%**.
Este paso filtra el período 2025-2026 para comparación directa.


In [ ]:
idx_2025 = [i for i, d in enumerate(d5) if d.year >= 2025]

if idx_2025:
    a_25 = a5[idx_2025]; p_25 = p5[idx_2025]
    m_25 = calcular_metricas(a_25, p_25, "Período 2025-2026")
    print("=" * 55)
    print("  PERÍODO 2025-2026 — comparable con SARIMA/STL")
    print("=" * 55)
    imprimir_metricas(m_25)
else:
    print("⚠️  No hay predicciones en 2025-2026 — todos los puntos ya están en ese período")
    m_25 = m5
    print("   Usando métricas del walk-forward completo")
    imprimir_metricas(m_25)


## PASO 5 — Split fijo vs Walk-Forward CV
**Split fijo:** train 2021-2024, test 2025-2026 (como Tamara y Álvaro)
**Walk-Forward CV:** predice cada mes con solo historia pasada (más riguroso)

Este paso muestra la diferencia entre ambos métodos para el mismo producto.


In [ ]:
# ─── Walk-Forward (ya calculado) ─────────────────────────
print("Walk-Forward CV:")
imprimir_metricas(m5)

print()

# ─── Split fijo ───────────────────────────────────────────
df_split = features_mensual(ts_ganador).dropna().reset_index(drop=True)
# Agregar la mejor feature
# df_split["temporada_sidra"] = df_split["fecha"].dt.month.isin([10, 11, 12]).astype(int)
# Descomentá la que haya mejorado
df_split = df_split.dropna().reset_index(drop=True)

a_sf, p_sf, d_sf = split_fijo(df_split, test_desde="2025-01-01")
m_sf = calcular_metricas(a_sf, p_sf, "Split Fijo (2025-2026)")
print("Split Fijo (train: 2021-2024 | test: 2025-2026):")
imprimir_metricas(m_sf)

print()
print("=" * 55)
print("  COMPARACIÓN")
print("=" * 55)
print(f"  Walk-Forward SMAPE: {m5['SMAPE']:.2f}%")
print(f"  Split Fijo   SMAPE: {m_sf['SMAPE']:.2f}%")
diff = m5["SMAPE"] - m_sf["SMAPE"]
if abs(diff) < 1:
    print(f"  🔵 Resultados similares (diff: {diff:+.2f}%)")
elif diff > 0:
    print(f"  ✅ Split fijo da mejor SMAPE por {diff:.2f}%")
    print(f"     Pero walk-forward es más confiable — no hay data leakage")
else:
    print(f"  ✅ Walk-forward da mejor SMAPE por {abs(diff):.2f}%")
    print(f"     Y además es metodológicamente más riguroso")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Sidra Preferida — Walk-Forward vs Split Fijo", fontsize=12, fontweight="bold")

# Walk-Forward
axes[0].plot(d5, a5, label="Real", color="#EF9F27", linewidth=2)
axes[0].plot(d5, p5, label="Predicho", color="#D85A30", linewidth=2, linestyle="--")
axes[0].set_title(f"Walk-Forward | SMAPE {m5['SMAPE']:.2f}%", fontsize=10)
axes[0].set_ylabel("Demanda"); axes[0].legend(); axes[0].grid(alpha=0.3)

# Split Fijo
if len(d_sf) > 0:
    axes[1].plot(d_sf, a_sf, label="Real", color="#EF9F27", linewidth=2)
    axes[1].plot(d_sf, p_sf, label="Predicho", color="#D85A30", linewidth=2, linestyle="--")
    axes[1].set_title(f"Split Fijo | SMAPE {m_sf['SMAPE']:.2f}%", fontsize=10)
    axes[1].legend(); axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "Sin datos en período 2025-2026",
                 ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout(); plt.show()
